# 🎬 YouTube Video Pipeline on Kaggle (GPU Accelerated)

Render video tự động (Kokoro TTS, Faster-Whisper CUDA, Remotion Render, Kinetic Captions, Thumbnail) trên **Kaggle GPU T4**.

In [ ]:
# [1] Kiểm tra GPU & Python
import os, sys, torch
print('Python:', sys.executable, sys.version)
print('CUDA Available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU Device:', torch.cuda.get_device_name(0))
os.environ['PYTHON_BIN'] = sys.executable

In [ ]:
# [2] Clone repo vào /tmp để /kaggle/working chỉ chứa file output sạch (MP4, SRT, PNG)
import subprocess, shutil

GITHUB_REPO = 'https://github.com/phamquanganhcode/ytb-pipeline.git'
GITHUB_BRANCH = 'feat/comic-explainer-en'
REPO_DIR = '/tmp/youtube-video-pipeline'

if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)
subprocess.run(['git', 'clone', '-b', GITHUB_BRANCH, '--single-branch', '--depth', '1', GITHUB_REPO, REPO_DIR], check=True)
os.chdir(REPO_DIR)
print('Working dir:', os.getcwd())

In [ ]:
# [3] Cài đặt môi trường (Node 20, FFmpeg, Chromium, Kokoro, Whisper)
subprocess.run(['bash', 'kaggle/setup_kaggle.sh'], check=True)

In [ ]:
# [4] Render Video (ComicExplainer EN)
SCRIPT = 'scripts/comic/binance-6b-hack-en.json'
print(f'=== Rendering: {SCRIPT} ===')
subprocess.run([sys.executable, 'build_comic.py', SCRIPT], check=True)

In [ ]:
# [5] Chỉ xuất MP4, SRT, Metadata, Thumbnail & Preview PNG ra /kaggle/working/
import glob

for pattern in (
    'out/binance_6b_comic_en.mp4',
    'out/binance_6b_comic_en.srt',
    'out/binance_6b_comic_en_youtube.txt',
    'out/binance_6b_comic_en_thumb.png',
    'out/binance_6b_comic_en_still_*.png',
):
    for f in glob.glob(pattern):
        dest = os.path.join('/kaggle/working', os.path.basename(f))
        shutil.copy2(f, dest)
        print('READY:', dest, os.path.getsize(dest), 'bytes')
print('DONE!')